In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load QCEW csv files
files = [
    "../data/QCEW_2018_q1-q4_Butte_County.csv",
    "../data/QCEW_2019_q1-q4_Butte_County.csv",
]
df_all = pd.concat([pd.read_csv(f) for f in files])

In [ ]:
# Filter just private supersectors
df = df_all[(df_all["agglvl_code"] == 74) & (df_all["own_code"] == 5)].copy()

In [ ]:
# Reshape to separate quarters into 3 separate month rows
value_vars = ["month1_emplvl", "month2_emplvl", "month3_emplvl"]
df = df.melt(
    id_vars=["year", "qtr", "industry_title"],
    value_vars=value_vars,
    var_name="month_num",
    value_name="employment",
)

# Convert months to real Datetime objects
df["month_idx"] = df["month_num"].str.extract("(\d)").astype(int)
df["actual_month"] = (df["qtr"] - 1) * 3 + df["month_idx"]
df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-" + df["actual_month"].astype(str) + "-01"
)

<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
/var/folders/q6/4k3737d52zb9_4wzscv_sx100000gn/T/ipykernel_36714/2253417036.py:11: SyntaxWarning: invalid escape sequence '\d'
  df['month_idx'] = df['month_num'].str.extract('(\d)').astype(int)


In [ ]:
df = df.sort_values(["industry_title", "date"])

In [ ]:
# Set fire dates
fire_start = pd.Timestamp("2018-11-08")
fire_end = pd.Timestamp("2018-11-25")

In [15]:
# Plot function
def plot_industry(name, data):
    plt.figure(figsize=(10, 5))
    plt.plot(data["date"], data["employment"], marker="o", color="#2c3e50", linewidth=2)

    # shade fire block (Novmeber 8th to November 25th)
    plt.axvspan(
        fire_start, fire_end, color="red", alpha=0.2, label="Camp Fire Duration"
    )
    plt.axvline(
        fire_start, color="red", linestyle="--", alpha=0.7, label="Started (Nov 8)"
    )
    plt.axvline(
        fire_end, color="orange", linestyle="--", alpha=0.7, label="Contained (Nov 25)"
    )

    plt.title(f"Employment impact in {name}", fontsize=14)
    plt.ylabel("Number of Employees")
    plt.xlabel("Date")
    plt.grid(True, axis="y", linestyle=":", alpha=0.7)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()

    # make file name
    clean_name = "".join([c if c.isalnum() else "_" for c in name])[:50]
    plt.savefig(f"../results/impact-on-industrys/Employment_{clean_name}.png")
    plt.close()

In [16]:
n = 0
for name, group in df.groupby("industry_title"):
    if group["employment"].notna().any():
        plot_industry(name, group)
        n += 1

print(f"{n} plots generated.")

20 plots generated.
